In [52]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score
from scipy.stats import skew
from scipy.stats import kurtosis

In [28]:
df=pd.read_csv('datasets/dataset1.csv')

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1205 entries, 0 to 1204
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dob                       1205 non-null   int64  
 1   sistolicki_krvni_tlak     1200 non-null   float64
 2   dijastolicki_krvni_tlak   1201 non-null   float64
 3   glukoza_u_krvi            1203 non-null   float64
 4   tjelesna_temp             1205 non-null   int64  
 5   BMI                       1187 non-null   float64
 6   komplikacije_u_proslosti  1203 non-null   float64
 7   dijabetes                 1203 non-null   float64
 8   gestacijski_dijabetes     1205 non-null   int64  
 9   mentalno_zdravlje         1205 non-null   int64  
 10  otkucaji_srca             1203 non-null   float64
 11  nivo_rizika               1187 non-null   object 
dtypes: float64(7), int64(4), object(1)
memory usage: 113.1+ KB


In [30]:
df.head()

,dob,sistolicki_krvni_tlak,dijastolicki_krvni_tlak,glukoza_u_krvi,tjelesna_temp,BMI,komplikacije_u_proslosti,dijabetes,gestacijski_dijabetes,mentalno_zdravlje,otkucaji_srca,nivo_rizika
0,22,90.0,60.0,9.0,100,18.0,1.0,1.0,0,1,80.0,High
1,22,110.0,70.0,7.1,98,20.4,0.0,0.0,0,0,74.0,Low
2,27,110.0,70.0,7.5,98,23.0,1.0,0.0,0,0,72.0,Low
3,20,100.0,70.0,7.2,98,21.2,0.0,0.0,0,0,74.0,Low
4,20,90.0,60.0,7.5,98,19.7,0.0,0.0,0,0,74.0,Low


In [31]:
df['tjelesna_temp'] = ((df['tjelesna_temp'] - 32) * 5/9).round(2)

In [32]:
df.head()

,dob,sistolicki_krvni_tlak,dijastolicki_krvni_tlak,glukoza_u_krvi,tjelesna_temp,BMI,komplikacije_u_proslosti,dijabetes,gestacijski_dijabetes,mentalno_zdravlje,otkucaji_srca,nivo_rizika
0,22,90.0,60.0,9.0,37.78,18.0,1.0,1.0,0,1,80.0,High
1,22,110.0,70.0,7.1,36.67,20.4,0.0,0.0,0,0,74.0,Low
2,27,110.0,70.0,7.5,36.67,23.0,1.0,0.0,0,0,72.0,Low
3,20,100.0,70.0,7.2,36.67,21.2,0.0,0.0,0,0,74.0,Low
4,20,90.0,60.0,7.5,36.67,19.7,0.0,0.0,0,0,74.0,Low


In [33]:
df['nivo_rizika'].unique()

array(['High', 'Low', nan], dtype=object)

In [34]:
print(df.isnull().sum()) 

dob                          0
sistolicki_krvni_tlak        5
dijastolicki_krvni_tlak      4
glukoza_u_krvi               2
tjelesna_temp                0
BMI                         18
komplikacije_u_proslosti     2
dijabetes                    2
gestacijski_dijabetes        0
mentalno_zdravlje            0
otkucaji_srca                2
nivo_rizika                 18
dtype: int64


In [37]:
varijable = [
    'sistolicki_krvni_tlak',
    'dijastolicki_krvni_tlak',
    'glukoza_u_krvi',
    'BMI',
    'komplikacije_u_proslosti',
    'dijabetes',
    'otkucaji_srca'
]


skewness_values = {}
for var in varijable:
    if var in df.columns:
       
        podaci = df[var].dropna()
        
        
        if pd.api.types.is_numeric_dtype(podaci):
            skewness_values[var] = skew(podaci)
        else:
            print(f"Varijabla '{var}' nije numerička. Skewness se ne računa.")
            skewness_values[var] = None
    else:
        print(f"Varijabla '{var}' ne postoji u DataFrame-u.")
        skewness_values[var] = None


print("\nKoeficijent iskrivljenosti (skewness):")
for var, sk in skewness_values.items():
    if sk is not None:
        print(f"{var}: {sk:.4f}")
    else:
        print(f"{var}: nije izračunat")


Koeficijent iskrivljenosti (skewness):
sistolicki_krvni_tlak: 0.2571
dijastolicki_krvni_tlak: 0.3772
glukoza_u_krvi: 1.5781
BMI: 0.4574
komplikacije_u_proslosti: 1.7071
dijabetes: 0.9339
otkucaji_srca: 0.2097


In [42]:
skewness_dict = {
    'sistolicki_krvni_tlak': 0.2571,
    'dijastolicki_krvni_tlak': 0.3772,
    'glukoza_u_krvi': 1.5781,
    'BMI': 0.4574,
    'komplikacije_u_proslosti': 1.7071,
    'dijabetes': 0.9339,
    'otkucaji_srca': 0.2097
}

for var, skew_val in skewness_dict.items():
    if var not in df.columns:
        print(f"Varijabla '{var}' ne postoji u DataFrame-u.")
        continue
    
    if skew_val < -1 or skew_val > 1:
        fill_value = df[var].median(skipna=True)
        method = "median"
    else:
        fill_value = df[var].mean(skipna=True)
        method = "mean"
    
    df[var] = df[var].fillna(fill_value)
    
    print(f"{var}: skew={skew_val:.4f} → popunjeno sa {method} = {fill_value:.4f}")

# Provjera
print("\nPreostale missing vrijednosti:")
print(df[list(skewness_dict.keys())].isnull().sum())

sistolicki_krvni_tlak: skew=0.2571 → popunjeno sa mean = 116.8324
dijastolicki_krvni_tlak: skew=0.3772 → popunjeno sa mean = 77.1759
glukoza_u_krvi: skew=1.5781 → popunjeno sa median = 6.9000
BMI: skew=0.4574 → popunjeno sa mean = 23.3104
komplikacije_u_proslosti: skew=1.7071 → popunjeno sa median = 0.0000
dijabetes: skew=0.9339 → popunjeno sa mean = 0.2880
otkucaji_srca: skew=0.2097 → popunjeno sa mean = 75.8174

Preostale missing vrijednosti:
sistolicki_krvni_tlak       0
dijastolicki_krvni_tlak     0
glukoza_u_krvi              0
BMI                         0
komplikacije_u_proslosti    0
dijabetes                   0
otkucaji_srca               0
dtype: int64


In [44]:
print(df['nivo_rizika'].value_counts(dropna=False))

nivo_rizika
Low     731
High    474
Name: count, dtype: int64


In [45]:
mod_value = df['nivo_rizika'].mode()[0]
df['nivo_rizika'] = df['nivo_rizika'].fillna(mod_value)
print(f"Popunjeno s modom: {mod_value}")

Popunjeno s modom: Low


In [46]:
encoder = LabelEncoder()

In [47]:
df['nivo_rizika'].unique()

array(['High', 'Low'], dtype=object)

In [48]:
df['nivo_rizika'] = df['nivo_rizika'].map({'High':1, 'Low':0})

In [49]:
df.head()

,dob,sistolicki_krvni_tlak,dijastolicki_krvni_tlak,glukoza_u_krvi,tjelesna_temp,BMI,komplikacije_u_proslosti,dijabetes,gestacijski_dijabetes,mentalno_zdravlje,otkucaji_srca,nivo_rizika
0,22,90.0,60.0,9.0,37.78,18.0,1.0,1.0,0,1,80.0,1
1,22,110.0,70.0,7.1,36.67,20.4,0.0,0.0,0,0,74.0,0
2,27,110.0,70.0,7.5,36.67,23.0,1.0,0.0,0,0,72.0,0
3,20,100.0,70.0,7.2,36.67,21.2,0.0,0.0,0,0,74.0,0
4,20,90.0,60.0,7.5,36.67,19.7,0.0,0.0,0,0,74.0,0


In [50]:
df.describe()

,dob,sistolicki_krvni_tlak,dijastolicki_krvni_tlak,glukoza_u_krvi,tjelesna_temp,BMI,komplikacije_u_proslosti,dijabetes,gestacijski_dijabetes,mentalno_zdravlje,otkucaji_srca,nivo_rizika
count,1205.000000,1205.000000,1205.000000,1205.000000,1205.000000,1205.000000,1205.000000,1205.000000,1205.000000,1205.00000,1205.000000,1205.000000
mean,27.731950,116.832365,77.175934,7.501064,36.889320,23.310373,0.175395,0.287967,0.117842,0.33444,75.817427,0.393361
std,12.571074,18.677721,14.282296,3.046988,0.603329,3.846792,0.380147,0.453004,0.322555,0.47199,7.221337,0.488699
min,10.000000,70.000000,40.000000,3.000000,36.110000,0.000000,0.000000,0.000000,0.000000,0.00000,58.000000,0.000000
25%,21.000000,100.000000,65.000000,6.000000,36.670000,21.000000,0.000000,0.000000,0.000000,0.00000,70.000000,0.000000
50%,25.000000,120.000000,80.000000,6.900000,36.670000,23.000000,0.000000,0.000000,0.000000,0.00000,76.000000,0.000000
75%,32.000000,130.000000,90.000000,7.900000,36.670000,25.000000,0.000000,1.000000,0.000000,1.00000,80.000000,1.000000
max,325.000000,200.000000,140.000000,19.000000,39.440000,37.000000,1.000000,1.000000,1.000000,1.00000,92.000000,1.000000


In [ ]:
varijable = [
    'sistolicki_krvni_tlak',
    'dijastolicki_krvni_tlak',
    'glukoza_u_krvi',
    'BMI',
    'komplikacije_u_proslosti',
    'dijabetes',
    'otkucaji_srca'
]

# Izračun kurtosis za svaku varijablu (ignorirajući NaN)
kurtosis_values = {}
for var in varijable:
    if var in df.columns:
        # Ukloni missing vrijednosti
        podaci = df[var].dropna()
        
        # Provjera jesu li podaci numerički
        if pd.api.types.is_numeric_dtype(podaci):
            # Fisher's kurtosis (excess) – normalna distribucija ima 0
            kurtosis_values[var] = kurtosis(podaci, fisher=True)
        else:
            print(f"Varijabla '{var}' nije numerička. Kurtosis se ne računa.")
            kurtosis_values[var] = None
    else:
        print(f"Varijabla '{var}' ne postoji u DataFrame-u.")
        kurtosis_values[var] = None

print("\nKoeficijent kurtosisa (Fisher, excess kurtosis):")
print("(0 = normalna distribucija, >0 = oštriji vrh, <0 = plosnatiji vrh)")
for var, kurt in kurtosis_values.items():
    if kurt is not None:
        print(f"{var}: {kurt:.4f}")
    else:
        print(f"{var}: nije izračunat")


Koeficijent kurtosisa (Fisher, excess kurtosis):
(0 = normalna distribucija, >0 = oštriji vrh, <0 = plosnatiji vrh)
sistolicki_krvni_tlak: 0.3637
dijastolicki_krvni_tlak: -0.2883
glukoza_u_krvi: 2.6309
BMI: 1.2839
komplikacije_u_proslosti: 0.9206
dijabetes: -1.1229
otkucaji_srca: -0.1682
